# The Mathematical Framework of Transformer Circuits  
## A very detailed companion notebook for AI learning and mechanistic interpretability

This notebook is a **deep study guide** for the paper **_A Mathematical Framework for Transformer Circuits_** (Anthropic / Transformer Circuits thread, 2021).

It is designed to be your **one-stop companion** before and during reading the original paper. The goal is not merely to restate the paper, but to make it **learnable**.

This notebook emphasizes:

- careful intuition
- explicit mathematical structure
- clean notation
- worked examples
- small executable Python snippets
- connections to mechanistic interpretability
- guidance on *what the paper is trying to do and why it matters*

---

## How to use this notebook

A good reading order is:

1. Read this notebook once without worrying about every symbol.
2. Skim the original paper and notice the big structural ideas.
3. Re-read the notebook and now connect each section to the paper.
4. Return to the paper's derivations with this notebook beside you.
5. Run the code cells and modify them.

---

## What this paper is really about

At a high level, the paper tries to do something extremely important:

> It tries to give us a language for describing what transformers compute in terms of **paths**, **linear maps**, **attention-mediated routing**, and **contributions to logits**.

That matters because mechanistic interpretability is not satisfied with vague statements like:

- “this head looks at previous tokens”
- “this neuron encodes syntax”
- “this layer represents meaning”

Instead, it wants to say things like:

- which component writes what into the residual stream
- which later component reads it
- how those interactions combine
- how that finally changes the logits

That is the spirit of the paper.

# Table of contents

1. Why this paper matters
2. What object a transformer acts on
3. The residual stream as the central state
4. Linear algebra refreshers you need
5. Embeddings, unembeddings, and logits
6. Attention written carefully
7. Why \(Q\), \(K\), and \(V\) are just learned linear maps
8. Residual connections and why they matter so much
9. Attention heads as read-write operations on the residual stream
10. The paper's simplifications
11. Zero-layer transformers
12. One-layer attention-only transformers
13. Two-layer attention-only transformers and head composition
14. Paths, virtual weights, and contribution accounting
15. Why the paper focuses on logits
16. Mechanistic interpretability viewpoint
17. Important caveats and limits of the framework
18. Worked examples in Python
19. How to read the original paper productively
20. Key takeaways
21. Further reading

# 1. Why this paper matters

This paper is important because it gave the field one of its earliest **clean mathematical decompositions** of transformer behavior.

It does **not** explain every aspect of modern large language models. Instead, it tries to build a tractable mathematical framework for understanding how transformer computations can be decomposed into understandable pieces.

A core ambition of mechanistic interpretability is:

> Can we describe model behavior as a composition of understandable computational subroutines?

This paper says: **let us start with a simplified setting where that question becomes manageable**.

The paper therefore makes several simplifying choices:

- attention-only transformers
- no MLPs
- no layer normalization in the main mathematical treatment
- small numbers of layers, especially 0, 1, and 2 layers
- a strong emphasis on tracing effects into the final logits

This may sound narrow, but it is actually powerful.

Why?

Because if you cannot understand a simplified transformer mathematically, you almost certainly cannot understand a large messy one.

---

## Big conceptual shift

A normal deep-learning explanation says:

- token embeddings go in
- attention mixes information
- output logits come out

But this paper says:

- the transformer maintains a **residual stream**
- each component **reads from** that stream
- computes something
- then **writes back to** that stream
- the final logits are a linear readout of that residual state

This “read/write to the residual stream” view is absolutely central for later mechanistic interpretability work.

# 2. What object does a transformer act on?

Suppose we have a sequence of \(n\) tokens and a model dimension \(d_{\text{model}}\).

After embedding and adding positional information, the sequence can be represented as a matrix:

$$
X \in \mathbb{R}^{n \times d_{\text{model}}}
$$

where each row corresponds to one token position.

So the transformer does **not** act on a single vector. It acts on a **sequence of vectors**.

We can write the token representation at position \(i\) as:

$$
x_i \in \mathbb{R}^{d_{\text{model}}}
$$

and the whole sequence as:

$$
X =
\begin{bmatrix}
x_1 \\
x_2 \\
\vdots \\
x_n
\end{bmatrix}
$$

This is already important.

A transformer is not simply learning a function

$$
f : \mathbb{R}^d \to \mathbb{R}^d
$$

but more like a structured function over a set of token-position states:

$$
F : \mathbb{R}^{n \times d_{\text{model}}} \to \mathbb{R}^{n \times d_{\text{model}}}
$$

and then finally to logits:

$$
\text{logits} \in \mathbb{R}^{n \times |\mathcal{V}|}
$$

where \(|\mathcal{V}|\) is the vocabulary size.

---

## Why sequence structure matters

Each position has two kinds of information mixed together:

1. **token identity information**
2. **position-dependent contextual information**

As layers proceed, the representation at a position is no longer “just the token”. It becomes:

- token meaning
- contextual clues
- copied features from other positions
- intermediate computation results
- evidence used later for prediction

That evolving per-position state is what mechanistic interpretability tries to understand.

# 3. The residual stream as the central state

The residual stream is arguably the most important object in the paper.

A residual block has the generic shape:

$$
x^{(\ell+1)} = x^{(\ell)} + F^{(\ell)}(x^{(\ell)})
$$

For a whole sequence, write:

$$
X^{(\ell+1)} = X^{(\ell)} + F^{(\ell)}(X^{(\ell)})
$$

This says the next state is the previous state plus an update.

The crucial interpretability viewpoint is:

> The residual stream is the shared communication channel of the model.

Every submodule:

- reads from it
- computes something
- writes a result back into it

That means the model is not a chain of completely separate hidden states. Instead, it is a sequence of modifications to a common evolving state.

---

## Why this is such a powerful framing

Suppose an attention head computes something useful, like “copy the previous name token”. It does **not** directly produce the final answer. Instead, it writes something into the residual stream. Then later heads or later layers can read that information and use it.

So a useful causal description often looks like:

1. some early component writes a feature
2. a later component notices or transforms that feature
3. a final component causes a logit increase for the correct token

This naturally leads to **circuits**.

A circuit is not just “a head with a role”. It is a **chain of interacting writes and reads** through the residual stream.

# 4. Linear algebra refreshers you need

Transformers are built largely from a few operations:

- matrix multiplication
- vector addition
- dot products
- softmax
- affine maps
- composition

Let us revisit each carefully.

## 4.1 Linear map

A linear map \(W\) satisfies:

$$
W(\alpha u + \beta v) = \alpha W u + \beta W v
$$

In matrix form:

$$
y = Wx
$$

Geometrically, a linear map can perform:

- scaling
- rotation
- reflection
- shearing
- projection

In neural nets, learned matrices are repeatedly changing representation coordinates.

---

## 4.2 Affine map

An affine map is:

$$
y = Wx + b
$$

This is a linear map plus translation.

---

## 4.3 Dot product

For vectors \(q, k \in \mathbb{R}^d\),

$$
q \cdot k = q^\top k = \sum_{i=1}^{d} q_i k_i
$$

This measures a kind of alignment or similarity.

Attention uses many dot products between query vectors and key vectors.

---

## 4.4 Matrix product as batched similarity

If \(Q \in \mathbb{R}^{n \times d_k}\) and \(K \in \mathbb{R}^{n \times d_k}\), then:

$$
QK^\top \in \mathbb{R}^{n \times n}
$$

and entry \((i,j)\) equals:

$$
(QK^\top)_{ij} = q_i^\top k_j
$$

So this matrix contains all query-key similarities.

---

## 4.5 Softmax

Given a vector \(z\),

$$
\text{softmax}(z)_i = \frac{e^{z_i}}{\sum_j e^{z_j}}
$$

This turns real-valued scores into positive weights summing to 1.

That means softmax produces something like a learned distribution over “where to attend”.

---

## 4.6 Weighted sum

If attention weights from token \(i\) to all positions are \(\alpha_{ij}\), then the output at position \(i\) is:

$$
o_i = \sum_{j=1}^{n} \alpha_{ij} v_j
$$

This is not just selecting one value. It is forming a **weighted mixture** of value vectors.

In [ ]:
import numpy as np

# Tiny demonstrations of the core operations
x = np.array([1.0, 2.0])
W = np.array([[2.0, 0.0],
              [0.0, -1.0]])
b = np.array([0.5, 0.5])

linear = W @ x
affine = W @ x + b
dot = np.array([1.0, 3.0]) @ np.array([4.0, -2.0])

def softmax(z):
    z = z - np.max(z)
    e = np.exp(z)
    return e / np.sum(e)

weights = softmax(np.array([2.0, 1.0, 0.0]))

print("Linear map W @ x:", linear)
print("Affine map W @ x + b:", affine)
print("Dot product:", dot)
print("Softmax weights:", weights, "sum =", weights.sum())

# 5. Embeddings, unembeddings, and logits

A language model begins with discrete tokens and must end with scores over the vocabulary.

This means there are two especially important matrices:

- the **embedding matrix**
- the **unembedding matrix**

Let the vocabulary size be \(|\mathcal{V}|\), and model dimension be \(d_{\text{model}}\).

## 5.1 Embedding

If token \(t\) is represented as a one-hot vector \(e_t \in \mathbb{R}^{|\mathcal{V}|}\), then its embedding is:

$$
x_t = W_E^\top e_t
$$

or equivalently, “look up row \(t\)” of the embedding table.

So:

$$
W_E \in \mathbb{R}^{|\mathcal{V}| \times d_{\text{model}}}
$$

---

## 5.2 Positional embedding

If position \(p\) has positional embedding \(p_p\), the starting representation is often:

$$
x^{(0)}_i = e_{\text{token}_i} W_E + p_i
$$

This matters because attention must know *which position* a token is in, not just what token it is.

---

## 5.3 Unembedding

At the end of the model, the residual vector at position \(i\), say \(r_i\), is mapped to logits:

$$
\ell_i = r_i W_U
$$

where

$$
W_U \in \mathbb{R}^{d_{\text{model}} \times |\mathcal{V}|}
$$

The entry \(\ell_{i,t}\) is the logit assigned to token \(t\) at position \(i\).

---

## 5.4 Why this matters for the paper

The paper often wants to say:

- what contribution does some head make
- to the final logit of a specific token?

Because the unembedding is linear, contributions in residual space can be propagated into logit space in a mathematically clean way.

This is one of the reasons the paper cares so much about **logit contributions**.

If a residual contribution is \(\Delta r_i\), then its contribution to logits is:

$$
\Delta \ell_i = \Delta r_i W_U
$$

That linearity is interpretability gold.

In [ ]:
# Tiny embedding / unembedding demo

vocab = ["the", "cat", "sat", "on", "mat"]
vocab_size = len(vocab)
d_model = 4

rng = np.random.default_rng(0)

W_E = rng.normal(size=(vocab_size, d_model))
W_U = rng.normal(size=(d_model, vocab_size))

token_id = vocab.index("cat")
one_hot = np.eye(vocab_size)[token_id]

embedding = one_hot @ W_E

# pretend this is the final residual vector at some position
residual = rng.normal(size=(d_model,))
logits = residual @ W_U

print("Token:", vocab[token_id])
print("Embedding shape:", embedding.shape)
print("Residual shape:", residual.shape)
print("Logits shape:", logits.shape)
print("Predicted vocabulary scores:")
for tok, score in zip(vocab, logits):
    print(f"  {tok:>3}: {score: .4f}")

# 6. Attention written carefully

The standard attention formula is

$$
\text{Attn}(Q,K,V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V
$$

Let us unpack every piece.

Suppose the sequence length is \(n\). Then:

- \(Q \in \mathbb{R}^{n \times d_k}\)
- \(K \in \mathbb{R}^{n \times d_k}\)
- \(V \in \mathbb{R}^{n \times d_v}\)

Then:

$$
S = \frac{QK^\top}{\sqrt{d_k}} \in \mathbb{R}^{n \times n}
$$

is the score matrix.

Entry \((i,j)\) is:

$$
S_{ij} = \frac{q_i^\top k_j}{\sqrt{d_k}}
$$

Then we apply softmax row-wise:

$$
A_{ij} = \frac{e^{S_{ij}}}{\sum_{m=1}^n e^{S_{im}}}
$$

So \(A\) is the attention pattern matrix.

Finally, the output is:

$$
O = AV
$$

and row \(i\) is

$$
o_i = \sum_{j=1}^{n} A_{ij} v_j
$$

---

## Interpretation

At position \(i\):

1. the query \(q_i\) asks: “what kind of information am I looking for?”
2. each key \(k_j\) says: “what information do I advertise at position \(j\)?”
3. the dot products compare the query to each key
4. softmax turns similarities into weights
5. the value vectors are mixed using those weights

So attention has a beautiful separation:

- **queries and keys** decide *where to look*
- **values** decide *what content is moved*

That separation becomes very important when thinking mechanistically.

# 7. Why \(Q\), \(K\), and \(V\) are just learned linear maps

A common beginner mistake is to think queries, keys, and values are mysterious special entities.

They are not. They are simply learned projections of the residual stream.

If the current residual stream matrix is \(X\), then for one head:

$$
Q = XW_Q,\quad K = XW_K,\quad V = XW_V
$$

where typically:

- \(W_Q \in \mathbb{R}^{d_{\text{model}} \times d_k}\)
- \(W_K \in \mathbb{R}^{d_{\text{model}} \times d_k}\)
- \(W_V \in \mathbb{R}^{d_{\text{model}} \times d_v}\)

This means a head is not reading the raw residual stream directly. It is reading **particular linear projections** of it.

That is already suggestive:

- the query projection chooses the subspace relevant to searching
- the key projection chooses the subspace relevant to matching
- the value projection chooses the subspace relevant to transmitted content

In mechanistic interpretability language, you can think of these as different “views” of the same residual state.

---

## Full one-head expression

Plugging the definitions into attention gives:

$$
\text{Head}(X) =
\text{softmax}\!\left(
\frac{XW_Q (XW_K)^\top}{\sqrt{d_k}}
\right) XW_V
$$

This looks complicated, but conceptually it is:

- compute match scores from \(Q\) and \(K\)
- use those scores to route value information
- write the result back through an output matrix

If there is an output matrix \(W_O\), then the head output written to the residual stream is:

$$
\text{HeadOut}(X) =
\text{softmax}\!\left(
\frac{XW_Q (XW_K)^\top}{\sqrt{d_k}}
\right) XW_V W_O
$$

Now the output is back in \(d_{\text{model}}\)-space and can be added to the residual stream.

In [ ]:
# Attention from scratch, carefully written

rng = np.random.default_rng(1)
n = 5
d_model = 6
d_k = 4
d_v = 4

X = rng.normal(size=(n, d_model))
W_Q = rng.normal(size=(d_model, d_k))
W_K = rng.normal(size=(d_model, d_k))
W_V = rng.normal(size=(d_model, d_v))
W_O = rng.normal(size=(d_v, d_model))

def row_softmax(M):
    M = M - M.max(axis=-1, keepdims=True)
    E = np.exp(M)
    return E / E.sum(axis=-1, keepdims=True)

Q = X @ W_Q
K = X @ W_K
V = X @ W_V

scores = (Q @ K.T) / np.sqrt(d_k)
A = row_softmax(scores)
head_output_before_write = A @ V
head_write = head_output_before_write @ W_O

print("X shape:", X.shape)
print("Q shape:", Q.shape)
print("K shape:", K.shape)
print("V shape:", V.shape)
print("Attention pattern shape:", A.shape)
print("Head write shape:", head_write.shape)
print("\nRow sums of attention pattern (should be 1):")
print(A.sum(axis=1))

# 8. Residual connections and why they matter so much

Residual connections are often introduced as an optimization trick that helps gradients flow.

That is true, but from an interpretability perspective they matter even more.

A residual update has the form:

$$
X^{(\ell+1)} = X^{(\ell)} + \Delta^{(\ell)}
$$

where \(\Delta^{(\ell)}\) is the output of a component or block.

This means:

- previous information stays available
- new information can be added incrementally
- later components can still access earlier computations
- the model can build layered feature accumulations

This is why the paper emphasizes the residual stream as a common state.

---

## Additive decomposition

If the final residual stream is built from many additive contributions:

$$
R_{\text{final}} = R_0 + \Delta_1 + \Delta_2 + \cdots + \Delta_m
$$

then final logits are:

$$
\ell = R_{\text{final}} W_U
= R_0 W_U + \Delta_1 W_U + \Delta_2 W_U + \cdots + \Delta_m W_U
$$

This means we can ask:

- how much of the final logit came from embedding?
- how much came from head \(h\)?
- how much came from a specific path through the network?

This is a major reason the framework is analytically useful.

# 9. Attention heads as read-write operations on the residual stream

A very productive mental model is:

> each attention head is a specialized read/write module.

Let the current residual at position \(i\) be \(r_i\).

Then for a head:

- the **query** projection of \(r_i\) decides what it wants
- the **keys** of all positions decide who matches
- the **values** of matched positions provide content
- the **output projection** writes that content back to the residual stream

So an attention head can be thought of as:

1. deciding *where to read from*
2. deciding *what to read*
3. deciding *how to write what it read back into model space*

This suggests common head behaviors like:

- previous-token heads
- induction heads
- name mover heads
- delimiter heads
- syntactic relation heads

The head itself is not “meaning”. It is a structured operator acting on representations.

# 10. The paper's simplifications

The paper deliberately simplifies the transformer.

This is essential to understand.

## 10.1 Attention-only

It removes MLPs and studies attention-only models.

Why?

Because MLPs are powerful but harder to analyze cleanly in circuit terms. Attention already introduces rich routing structure, so it is a natural first target.

## 10.2 No LayerNorm in the main framework

LayerNorm introduces nontrivial nonlinear, sequence-coupled effects that complicate clean linear path decompositions.

Removing it makes the algebra cleaner.

## 10.3 No biases in some formulations

Biases can often be absorbed by augmenting the state with an extra constant dimension, so they are omitted for clarity.

## 10.4 Small numbers of layers

The paper focuses especially on:

- 0-layer transformers
- 1-layer attention-only transformers
- 2-layer attention-only transformers

Why those?

Because once you understand what 0, 1, and 2 layers can compute, you begin to see:

- what individual heads do
- what one layer can express
- what head composition across layers enables

That composition is one of the most important themes in the paper.

# 11. Zero-layer transformers

A zero-layer transformer means there are no attention layers.

So what remains?

- token embedding
- positional embedding
- perhaps direct unembedding to logits

In the simplified viewpoint, the model is predicting from the current token representation without contextual interaction.

That means the next-token prediction is driven by a mapping like:

$$
\ell_i = x_i W_U
$$

where \(x_i\) mainly depends on the token at the current position and its position encoding.

Since there is no interaction among positions, a zero-layer language model is essentially a kind of **bigram model** in spirit: prediction depends only on the current token identity (plus position features, if any).

This is the first conceptual anchor.

---

## Why this matters

It gives us a baseline:

- no attention means no contextual copying
- no cross-position communication
- no multi-token algorithmic behavior

So any behavior beyond local token statistics must emerge once attention is added.

In [ ]:
# A tiny "zero-layer" style toy example
# We pretend the next-token logit depends only on the current token embedding.

vocab = ["A", "B", "C", "D"]
vocab_size = len(vocab)
d_model = 3
rng = np.random.default_rng(2)

W_E = rng.normal(size=(vocab_size, d_model))
W_U = rng.normal(size=(d_model, vocab_size))

for tok in vocab:
    token_id = vocab.index(tok)
    x = W_E[token_id]              # token embedding only
    logits = x @ W_U
    pred = vocab[np.argmax(logits)]
    print(f"Current token {tok} -> predicted next token {pred}, logits = {np.round(logits, 3)}")

# 12. One-layer attention-only transformers

Now we add one layer of attention.

At position \(i\), the output can depend on all earlier positions (for causal attention) through a weighted sum of values.

This means the model can now implement patterns more expressive than simple bigram statistics.

A one-layer attention-only transformer can often be understood as learning a family of context-dependent templates like:

- if I see token \(A\), attend to a previous \(B\)
- copy a feature from some matching earlier token
- use that copied feature to change the current logit

The paper characterizes one-layer behavior in terms that are richer than pure bigrams, including patterns often described informally as skip-trigram-like effects.

The main lesson is:

> one attention layer introduces one step of cross-position information movement.

That sounds simple, but it is already enough to implement useful contextual operations.

---

## Why one layer is still limited

With only one layer:

- a head can read directly from the current residual state
- but it cannot easily depend on information that itself required a previous attention-mediated computation from another layer

So one-layer transformers are useful but do not yet exhibit the more powerful compositional behavior that emerges with multi-layer circuits.

# 13. Two-layer attention-only transformers and head composition

This is one of the deepest ideas in the paper.

With two layers, a head in layer 2 can read information that was written by a head in layer 1.

That means the computation is no longer just “directly attend to some source token”.

Instead, we can have **composition**:

1. a first-layer head computes and writes an intermediate feature
2. a second-layer head reads that intermediate feature
3. the second-layer head uses it to route or compute something more sophisticated

This opens the door to algorithmic behaviors.

In informal mechanistic-interpretability language:

- layer-1 heads can set up useful markers or copied features
- layer-2 heads can exploit those markers

That is the essence of head composition.

---

## Functional picture

Let \(H^{(1)}\) be a head in layer 1 and \(H^{(2)}\) a head in layer 2.

Then:

$$
R^{(1)} = R^{(0)} + H^{(1)}(R^{(0)})
$$

and

$$
R^{(2)} = R^{(1)} + H^{(2)}(R^{(1)})
$$

Now \(H^{(2)}\) depends on a state that already includes \(H^{(1)}\)'s write.

So even if each head is individually simple, **their composition can be much more powerful**.

This idea later becomes crucial for understanding induction heads and other circuits.

# 14. Paths, virtual weights, and contribution accounting

One of the paper's most valuable conceptual contributions is the idea that we can analyze behavior in terms of **paths** through the computation.

Very loosely, a path might look like:

- embedding of token at position \(j\)
- transformed by value matrix of a head
- routed by attention pattern
- written into the residual stream at position \(i\)
- read out through the unembedding into a specific logit

The details can become algebraically involved, but the guiding idea is simple:

> If the whole system is composed of many linear maps plus attention-based routing, then we can ask how influence travels along specific routes.

This is why the phrase “transformer circuits” is apt. It is trying to identify functional pathways, not just activations in isolation.

---

## Virtual weights intuition

In some simplified cases, we can combine multiple linear maps into an effective or “virtual” weight.

For example, if a value vector is produced by \(W_V\), then projected back by \(W_O\), then read out by \(W_U\), the whole chain may be thought of as an effective map:

$$
W_{VUO\!U} = W_V W_O W_U
$$

Up to shape conventions and context, the point is that multiple linear steps can often be fused into one effective transformation.

This makes it easier to describe what a component is doing relative to logits.

---

## Why paths matter so much

Suppose two different heads both increase the logit for the same output token. That does not mean they are doing the same thing.

They may operate through very different paths:

- one might move token identity information
- another might move delimiter information
- another might create a match signal for a later head

A path-based analysis helps distinguish these possibilities.

# 15. Why the paper focuses on logits

In mechanistic interpretability, it is tempting to stop at a nice activation pattern and say “this head recognizes X”.

But the paper pushes toward a stricter standard:

> show how a component changes the final prediction

The final prediction comes from logits.

If a component writes \(\Delta r_i\) into the residual stream at position \(i\), then the logit contribution is:

$$
\Delta \ell_i = \Delta r_i W_U
$$

This means we can ask concrete questions like:

- which token logits are increased?
- which are suppressed?
- by how much?
- due to which head or path?

This is much more satisfying than vague semantic labeling.

---

## Logit lens connection

Later work often uses the **logit lens** idea: take an intermediate residual vector and project it through \(W_U\) as if it were already final.

That can reveal what token predictions are “latent” in the current state.

This paper's emphasis on linear readout into logits fits naturally with that way of thinking.

# 16. Mechanistic interpretability viewpoint

This paper is foundational because it encourages a specific decomposition of transformer computation.

## 16.1 Components are not isolated meanings

A head is not simply “the plural head” or “the syntax head”.

Instead, a head is an operator that:

- reads certain features
- uses those features to choose sources
- transports certain value-space content
- writes transformed content back to the residual stream

That is a much more mechanical description.

## 16.2 The residual stream is shared workspace

It is often helpful to think of the residual stream as a shared blackboard.

Different submodules write notes on the blackboard. Later submodules read those notes and add more notes.

The final unembedding reads the accumulated board contents and converts them to logits.

## 16.3 Circuits are chains of dependency

A circuit usually means a set of components whose interactions explain a behavior.

That often involves:

- a source feature
- one or more transport or transformation steps
- a final output effect

The paper helps formalize how to describe such chains.

# 17. Important caveats and limits of the framework

This is a powerful paper, but it is not the final word on transformers.

It is important to understand its limits.

## 17.1 Attention-only is not full transformer behavior

Real transformers also contain MLPs, and MLPs matter a lot.

Many concepts, features, and internal computations are strongly mediated by MLP layers.

## 17.2 LayerNorm matters

LayerNorm changes the effective geometry of the computation and introduces dependencies not captured by a purely linear-additive residual story.

## 17.3 Real models are larger and messier

Large language models have:

- many layers
- many heads
- richer nonlinear interactions
- polysemantic features
- superposition
- training artifacts
- distributional effects

So the framework is best seen as a foundational interpretability language, not a complete practical recipe for all modern models.

## 17.4 Attention patterns alone are not enough

A common trap is to look only at attention maps.

But attention weights alone do not tell you:

- what was moved
- whether the moved content mattered
- how it affected logits

You must also care about values, output projections, residual interactions, and downstream usage.

# 18. Worked examples in Python

The next few cells are not trying to reproduce the full paper. Instead, they illustrate the core mathematical ideas in small concrete forms.

We will show:

1. a tiny causal attention computation
2. additive residual updates
3. logit contributions from component writes
4. a toy illustration of two-step composition

In [ ]:
import numpy as np

rng = np.random.default_rng(3)

def row_softmax(M):
    M = M - M.max(axis=-1, keepdims=True)
    E = np.exp(M)
    return E / E.sum(axis=-1, keepdims=True)

# Tiny causal attention example
n = 4
d_model = 5
d_k = 3
d_v = 3

X = rng.normal(size=(n, d_model))
W_Q = rng.normal(size=(d_model, d_k))
W_K = rng.normal(size=(d_model, d_k))
W_V = rng.normal(size=(d_model, d_v))
W_O = rng.normal(size=(d_v, d_model))

Q = X @ W_Q
K = X @ W_K
V = X @ W_V

scores = (Q @ K.T) / np.sqrt(d_k)

# causal mask: forbid attending to future positions
mask = np.triu(np.ones((n, n)), k=1).astype(bool)
scores_masked = scores.copy()
scores_masked[mask] = -1e9

A = row_softmax(scores_masked)
head_out = (A @ V) @ W_O
X_next = X + head_out

print("Causal attention pattern:")
print(np.round(A, 3))
print("\nRow sums:")
print(np.round(A.sum(axis=1), 6))
print("\nResidual update shape:", head_out.shape)
print("Next residual state shape:", X_next.shape)

In [ ]:
# Additive decomposition of logit contributions

vocab = ["A", "B", "C", "D", "E"]
vocab_size = len(vocab)
W_U = rng.normal(size=(d_model, vocab_size))

residual_base = rng.normal(size=(d_model,))
delta_head1 = rng.normal(size=(d_model,))
delta_head2 = rng.normal(size=(d_model,))

final_residual = residual_base + delta_head1 + delta_head2

logits_base = residual_base @ W_U
logits_h1 = delta_head1 @ W_U
logits_h2 = delta_head2 @ W_U
logits_total = final_residual @ W_U

print("Does linear decomposition hold?")
print(np.allclose(logits_total, logits_base + logits_h1 + logits_h2))
print("\nPer-token total logits and contributions:")
for i, tok in enumerate(vocab):
    print(
        f"{tok}: total={logits_total[i]: .4f}, "
        f"base={logits_base[i]: .4f}, "
        f"h1={logits_h1[i]: .4f}, "
        f"h2={logits_h2[i]: .4f}"
    )

In [ ]:
# Toy two-step composition illustration

# Layer 1 writes an intermediate feature
R0 = rng.normal(size=(n, d_model))
intermediate_write = rng.normal(size=(n, d_model))
R1 = R0 + intermediate_write

# Layer 2 reads the modified state and writes again
second_write = 0.5 * R1   # not real attention, just a toy dependence
R2 = R1 + second_write

print("Norm of initial state:", np.linalg.norm(R0))
print("Norm after layer-1 write:", np.linalg.norm(R1))
print("Norm after layer-2 write:", np.linalg.norm(R2))

print("\nThis toy example is only to show the idea:")
print("Layer 2 is using a state that already includes layer 1's write.")

# 19. How to read the original paper productively

This paper can feel dense if you try to read it line by line the first time.

Here is a better strategy.

## First pass: understand the agenda

Ask:

- What simplifications are they making?
- Why do they care about the residual stream?
- Why are logits central?
- What are they claiming about 0, 1, and 2 layers?

Do not get stuck on every derivation yet.

## Second pass: track the computation objects

Keep asking:

- What space are we in right now?
- Is this vector in token space, residual space, value space, or logit space?
- Is this matrix a projection, a routing object, or a readout object?

This is where many readers get confused.

## Third pass: follow one effect end-to-end

Pick one concrete story:

- some token feature
- some head that moves it
- some later effect on logits

That is how you turn the paper from symbols into mechanism.

## Fourth pass: connect it to modern mech interp

Relate the paper to later concepts:

- induction heads
- logit lens
- path patching
- activation patching
- residual stream analysis
- superposition
- sparse autoencoders

That is where the paper becomes alive.

# 20. Key takeaways

Here are the most important ideas to carry forward.

## Takeaway 1
A transformer can be fruitfully viewed as a system of modules that **read from and write to the residual stream**.

## Takeaway 2
Attention is not just “paying attention”. It is a structured routing mechanism:

- queries determine what is sought
- keys determine where it is found
- values determine what content is moved

## Takeaway 3
Because the residual stream is additive and the unembedding is linear, we can often decompose final logits into contributions from components and paths.

## Takeaway 4
A one-layer attention-only model can already do contextual computation, but two layers enable **composition**, which is dramatically more expressive.

## Takeaway 5
The framework is simplified, but that simplification is a strength. It provides a clean mathematical language that later mechanistic-interpretability work builds on.

# 21. Further reading

After this notebook and the original paper, a strong next reading sequence is:

1. **Attention Is All You Need** — Vaswani et al.  
   Understand the original transformer architecture.

2. **A Walkthrough of A Mathematical Framework for Transformer Circuits** — Neel Nanda  
   Helpful interpretive commentary on the paper.

3. **Interpretability in the Wild / Transformer Circuits materials**  
   For the broader circuit-analysis mindset.

4. **Induction Heads** work  
   To see a concrete and famous example of head composition.

5. **Anthropic superposition / toy models of superposition**  
   For the feature-overlap problem that makes real models harder than simplified ones.

6. **Residual stream–centric interpretability resources**  
   Because the residual stream viewpoint is one of the most important habits to build.

---

## Final mental model

If you want one sentence to remember, keep this one:

> A transformer is a layered system in which components repeatedly read from a shared residual workspace, route and transform information, and write back updates that ultimately change the output logits.

That sentence is not the whole paper. But if you deeply understand it, the paper becomes much easier to read.

In [ ]:
# Optional: a tiny exercise section for self-study

questions = [
    "1. Why does the linearity of the unembedding make logit attribution easier?",
    "2. What is the difference between attention weights and value content?",
    "3. Why is the residual stream a better central object than focusing only on attention maps?",
    "4. What new capability appears when moving from one layer to two layers?",
    "5. Why are the paper's simplifications useful rather than merely unrealistic?"
]

print("Reflection / study questions:\n")
for q in questions:
    print(q)